libraries

In [1]:
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

print("Libraries loaded!")

Libraries loaded!


Dataset load

In [2]:
df = pd.read_csv("intents_large.csv")
print("Data loaded!")
print(df["intent"].value_counts())

Data loaded!
intent
greeting        40
bye             40
price           40
help            40
order           40
order_status    40
return          40
hours           40
location        40
thanks          40
about           40
Name: count, dtype: int64


In [3]:
X = df["text"]
y = df["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Vectorizer ready!")

Vectorizer ready!


Models

In [4]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline                            #Naive Bayes
from sklearn.feature_extraction.text import TfidfVectorizer

nb_improved = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=1)),
    ('model', MultinomialNB(alpha=0.1))
])

nb_improved.fit(X_train, y_train)
nb_imp_preds = nb_improved.predict(X_test)
print(f" Naive Bayes: {accuracy_score(y_test, nb_imp_preds) * 100:.1f}%")

 Naive Bayes: 85.2%


In [5]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression

lr_improved = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=1)),
    ('model', LogisticRegression(max_iter=500, C=10))
])

lr_improved.fit(X_train, y_train)
lr_imp_preds = lr_improved.predict(X_test)
print(f" Logistic Regression: {accuracy_score(y_test, lr_imp_preds) * 100:.1f}%")

 Logistic Regression: 80.7%


In [6]:
#Random Forest
from sklearn.ensemble import RandomForestClassifier

rf_improved = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=1)),
    ('model', RandomForestClassifier(n_estimators=200, max_depth=None))
])

rf_improved.fit(X_train, y_train)
rf_imp_preds = rf_improved.predict(X_test)
print(f" Random Forest: {accuracy_score(y_test, rf_imp_preds) * 100:.1f}%")

 Random Forest: 70.5%


In [7]:
#LinearSVC
from sklearn.svm import LinearSVC

svc_improved = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=1)),
    ('model', LinearSVC(C=10, max_iter=2000))
])

svc_improved.fit(X_train, y_train)
svc_imp_preds = svc_improved.predict(X_test)
print(f"Improved LinearSVC: {accuracy_score(y_test, svc_imp_preds) * 100:.1f}%")

Improved LinearSVC: 83.0%


In [8]:

results = {
    "Naive Bayes":          accuracy_score(y_test, nb_imp_preds),
    "Logistic Regression":  accuracy_score(y_test, lr_imp_preds),
    "Random Forest":        accuracy_score(y_test, rf_imp_preds),
    "LinearSVC":            accuracy_score(y_test, svc_imp_preds),
}

best_name = max(results, key=results.get)
best_models = {
    "Naive Bayes":         nb_improved,
    "Logistic Regression": lr_improved,
    "Random Forest":       rf_improved,
    "LinearSVC":           svc_improved,
}

best_model = best_models[best_name]
print(f"Best model: {best_name} ({results[best_name]*100:.1f}%)")

with open("best_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Best model saved!")

Best model: Naive Bayes (85.2%)
Best model saved!


In [ ]:
from groq import Groq

from dotenv import load_dotenv
import os

load_dotenv()
client = Groq(api_key=os.getenv("api_key"))

def get_response(user_input):
    intent = best_model.predict([user_input.lower()])[0]

    prompt = f"""You are a helpful customer support chatbot for a shop in Bhopal.
Customer said: "{user_input}"
Detected intent: "{intent}"
Reply in a short, friendly, natural way (2-3 sentences max).
Shop details:
- Pricing: Basic Rs 499, Pro Rs 999, Premium Rs 1999
- Returns within 7 days, refund in 3-5 days
- Open Monday to Saturday, 9 AM to 6 PM
- Located at MG Road, Bhopal"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=150
    )

    reply = response.choices[0].message.content
    return reply, intent

print("Chatbot ready! Type 'quit' to stop.\n")

while True:
    user = input("You: ").strip()
    if user.lower() == "quit":
        print("Bot: Goodbye!")
        break
    if not user:
        continue
    reply, intent = get_response(user)
    print(f"Bot [{intent}]: {reply}\n")

Chatbot ready! Type 'quit' to stop.

Bot [greeting]: Namaste. Welcome to our shop in Bhopal, located on MG Road. How can I assist you today?

Bot: Goodbye!
